In [1]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time

In [2]:
# Initialize data storage for plotting
timestamps = []
water_temperatures = []
ph_levels = []
turbidities = []
dissolved_oxygen_levels = []

In [3]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True,
        group_id="water_quality_processors",
        session_timeout_ms    = 30000,   # era el defecto 10000 → demasiado corto
        heartbeat_interval_ms = 5000,    # señal de vida cada 5s
        max_poll_interval_ms  = 300000,  # tiempo max entre polls
    )
    return consumer

In [4]:
sensor_history = {}

# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            s_id = sensor_data.get('sensor_id', 'default')
            if s_id not in sensor_history:
                sensor_history[s_id] = {
                    'timestamps': [], 'water_temperatures': [], 
                    'ph_levels': [], 'turbidities': [], 'dissolved_oxygen_levels': []
                }

            hist = sensor_history[s_id]

            # Update data storage
            hist['timestamps'].append(sensor_data['timestamp'])
            hist['water_temperatures'].append(sensor_data['water_temperature'])
            hist['ph_levels'].append(sensor_data['ph_level'])
            hist['turbidities'].append(sensor_data['turbidity'])
            hist['dissolved_oxygen_levels'].append(sensor_data['dissolved_oxygen'])

            # Keep only the last 100 entries for plotting
            if len(hist['timestamps']) > 100:
                hist['timestamps'].pop(0)
                hist['water_temperatures'].pop(0)
                hist['ph_levels'].pop(0)
                hist['turbidities'].pop(0)
                hist['dissolved_oxygen_levels'].pop(0)

            # Clear the current axes and redraw the plots
            plt.figure(figsize=(10, 8))

            for s, data in sensor_history.items():
                plt.subplot(2, 2, 1)
                plt.plot(data['timestamps'], data['water_temperatures'], label=f"Sensor {s}")
                plt.title("Water Temperature")
                plt.ylabel("°C")

                plt.subplot(2, 2, 2)
                plt.plot(data['timestamps'], data['ph_levels'], label=f"Sensor {s}")
                plt.title("pH Level")
                plt.ylabel("pH")

                plt.subplot(2, 2, 3)
                plt.plot(data['timestamps'], data['turbidities'], label=f"Sensor {s}")
                plt.title("Turbidity")
                plt.ylabel("NTU")

                plt.subplot(2, 2, 4)
                plt.plot(data['timestamps'], data['dissolved_oxygen_levels'], label=f"Sensor {s}")
                plt.title("Dissolved Oxygen")
                plt.ylabel("mg/L")

            plt.tight_layout()

            # Save the plot as an image (separado por partición para evitar sobreescritura)
            plt.savefig(f"water_quality_plot_{message.partition}.png")
            plt.close()

    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [ ]:
consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        update_plot(consumer)
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
Received: {'sensor_id': '3', 'timestamp': 1772331147, 'water_temperature': 30.90280151671662, 'ph_level': 8.855179864364114, 'turbidity': 28.28, 'dissolved_oxygen': 5.3}
Received: {'sensor_id': '1', 'timestamp': 1772331149, 'water_temperature': 32.2403673742995, 'ph_level': 8.452737196312139, 'turbidity': 40.42, 'dissolved_oxygen': 7.91}
Received: {'sensor_id': '3', 'timestamp': 1772331151, 'water_temperature': 32.0013979832892, 'ph_level': 8.278261255284207, 'turbidity': 24.58, 'dissolved_oxygen': 6.79}
Received: {'sensor_id': '1', 'timestamp': 1772331153, 'water_temperature': 31.33559814330181, 'ph_level': 8.791279246910591, 'turbidity': 17.42, 'dissolved_oxygen': 6.3}
Received: {'sensor_id': '3', 'timestamp': 1772331155, 'water_temperature': 30.607599096127615, 'ph_level': 8.549859702021342, 'turbidity': 29.64, 'dissolved_oxygen': 5.65}
Received: {'sensor_id': '1', 'timestamp': 1772331157, 'water_temperature': 31.4857885747381, 'ph_level': 